<a href="https://colab.research.google.com/github/Abdoulaye279/Financial-Engineering/blob/main/market-signals/01_direction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Direction of the S&P 500 the next day: two models against "always up"

Daily S&P 500, 2001-2026. Logistic regression and random forest on lagged returns, volatility, momentum and drawdown, refitted every January on all past data and tested 2010-2026. The baseline is not a coin flip but "always up", and the comparison is a paired test on the same days. Takes ~30 s.

In [1]:
# colab: get the repo first
try:
    import google.colab
    !git clone -q https://github.com/Abdoulaye279/Financial-Engineering
    %cd Financial-Engineering/market-signals
except ImportError:
    pass

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from signals import *
plt.style.use("quant.mplstyle")

spx = load_close("../black-scholes-sp500/data/sp500.csv")
r = np.log(spx).diff().dropna()          # daily log returns
print(len(r), r.index[0].date(), "->", r.index[-1].date())

6761 2000-01-04 -> 2026-11-19


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = features(r)
y = (r.shift(-1) > 0).astype(int)        # 1 = up tomorrow
ok = X.notna().all(axis=1) & r.shift(-1).notna()
X, y = X[ok], y[ok]
print(X.shape, X.index[0].date(), "->", X.index[-1].date())
print(f"up days: {y.mean():.1%}")
# 53.9% of days are up: a model that always says "up" is already right more than half the time,
# so 50% is the wrong benchmark

(6509, 9) 2001-01-02 -> 2026-11-18
up days: 53.9%


In [4]:
models = {
    "logistic": lambda: make_pipeline(StandardScaler(), LogisticRegression()),
    "random forest": lambda: RandomForestClassifier(300, min_samples_leaf=100, random_state=0, n_jobs=-1),
}
pred = {nm: walk_forward(X, y, mk, 2010) for nm, mk in models.items()}
yt = y[pred["logistic"].index]
right_up = yt == 1                       # "always up" is right on every up day
print(f"test 2010-2026: {len(yt)} days, always up {right_up.mean():.2%}")
for nm, p in pred.items():
    right = p == yt
    a, b, pv = mcnemar_exact(right, right_up)
    print(f"{nm:14s} accuracy {right.mean():.2%}  says up {p.mean():.0%} of days | "
          f"right when always-up is wrong: {a}, wrong when always-up is right: {b}, p = {pv:.2f}")
# logistic: 54.6% against 54.5% for always-up. it says "up" 94% of the time, so it is mostly a copy of the baseline
# random forest: 53.3%, below always-up. the paired test only looks at the days where the two disagree:
# 318 vs 368, p = 0.06, so if anything it is worse
# a test against 50% would have called both "significant" (p < 0.001 with 4,246 days): wrong question

test 2010-2026: 4246 days, always up 54.45%
logistic       accuracy 54.62%  says up 94% of days | right when always-up is wrong: 123, wrong when always-up is right: 116, p = 0.70
random forest  accuracy 53.27%  says up 84% of days | right when always-up is wrong: 318, wrong when always-up is right: 368, p = 0.06


In [5]:
# by year: is there any period where a model clearly beats always-up?
by_year = pd.DataFrame({nm: (p == yt).groupby(yt.index.year).mean() for nm, p in pred.items()})
by_year["always up"] = right_up.groupby(yt.index.year).mean()
by_year.style.format("{:.1%}")
# no stable edge: the random forest beats always-up in 2014, 2016, 2020-22 and 2024 and loses in the other years
# the best years for the models are the years with most down days (2020, 2022), the only ones where saying
# "down" can pay

,logistic,random forest,always up
Date,,,
2010,57.5%,55.6%,57.1%
2011,53.2%,49.6%,54.8%
2012,53.6%,49.2%,52.8%
2013,57.9%,54.4%,57.9%
2014,57.1%,59.1%,57.1%
2015,46.4%,45.6%,47.2%
2016,53.2%,52.8%,52.4%
2017,57.0%,53.4%,57.0%
2018,54.2%,51.8%,52.6%


In [6]:
# the economic test: hold the index when the model says up, 3-month T-bill otherwise (no costs)
rf = (load_close("data/irx.csv") / 100 / 252).reindex(r.index).ffill()   # ^IRX, annual % -> daily
rn, rft = r.shift(-1)[yt.index], rf.shift(-1)[yt.index]
for nm, p in pred.items():
    s = np.where(p == 1, rn, rft)
    z, pv = sharpe_test(s - rft, rn - rft)
    print(f"{nm:14s} Sharpe {sharpe(s, rft):.2f} vs buy and hold {sharpe(rn, rft):.2f}, "
          f"in the market {p.mean():.0%} of days, p = {pv:.2f}")
# logistic 0.68 vs 0.55 over the T-bill: it stays out on 6% of days
# p = 0.14, not significant over 17 years, and without trading costs
# accuracy and money don't say the same thing: a model can be right on small days and wrong on big ones

logistic       Sharpe 0.68 vs buy and hold 0.55, in the market 94% of days, p = 0.14
random forest  Sharpe 0.57 vs buy and hold 0.55, in the market 84% of days, p = 0.86


In [7]:
# with trading costs: each switch between the index and T-bills costs c (one way)
for c in [0.0005, 0.001]:
    for nm, p in pred.items():
        pos = p.values
        trades = np.abs(np.diff(np.r_[1, pos]))          # starts invested, like buy and hold
        s = np.where(pos == 1, rn, rft) - c * trades
        z, pv = sharpe_test(s - rft, rn - rft)
        print(f"{c * 1e4:.0f} bp  {nm:14s} trades/yr {trades.sum() / (len(pos) / 252):5.1f}  Sharpe {sharpe(s, rft):.2f} "
              f"vs buy and hold {sharpe(rn, rft):.2f}, p = {pv:.2f}")
# the logistic model trades 22 times a year: 0.61 at 5 bp, 0.54 at 10 bp, i.e. no better than buy and hold
# the random forest trades 54 times a year: 0.40 at 5 bp (p = 0.12) and 0.23 at 10 bp, significantly worse (p < 0.01)
# the small gain before costs does not survive them

5 bp  logistic       trades/yr  22.3  Sharpe 0.61 vs buy and hold 0.55, p = 0.48
5 bp  random forest  trades/yr  53.9  Sharpe 0.40 vs buy and hold 0.55, p = 0.12


10 bp  logistic       trades/yr  22.3  Sharpe 0.54 vs buy and hold 0.55, p = 0.93
10 bp  random forest  trades/yr  53.9  Sharpe 0.23 vs buy and hold 0.55, p = 0.00


In [8]:
# no cost model needed for the conclusion: the break-even cost per switch at which the Sharpe falls to buy and hold
from scipy.optimize import brentq
for nm, p in pred.items():
    pos = p.values
    trades = np.abs(np.diff(np.r_[1, pos]))
    gap = lambda c: sharpe(np.where(pos == 1, rn, rft) - c * trades, rft) - sharpe(rn, rft)
    c0 = brentq(gap, -0.01, 0.01)
    print(f"{nm:14s} break-even {c0 * 1e4:5.1f} bp per switch ({trades.sum() / (len(pos) / 252):.0f} switches a year)")
# logistic: any all-in cost above 9.4 bp per switch (spread, impact, tax on short-term gains) wipes out its edge
# random forest: 0.5 bp, below the spread of the most liquid S&P 500 ETF. whatever the cost model, it loses

logistic       break-even   9.4 bp per switch (22 switches a year)
random forest  break-even   0.5 bp per switch (54 switches a year)
